In [1]:
!pip install wordfreq
!pip install sentence-transformers==2.2.2
import re
from scipy.stats import zscore
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import re
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, models, util
from nltk.corpus import stopwords
from wordfreq import top_n_list
from nltk.corpus import stopwords
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - \ done
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=ff90c92991441466681fb67ab664b2871908506f653fbd4bd38bc75767bdbf64
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence-transformers


In [2]:
def get_embeddings(text, model):
    text = [str(t) for t in text]
    corpus_embeddings = model.encode(text, batch_size=32, show_progress_bar=True, convert_to_tensor=True)
    return corpus_embeddings

def get_embeddings_batched(texts, model, batch_size=32):
    """
    Computes embeddings for a list of texts in batches to save memory.
    """
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding batches"):
        batch_texts = texts[i:i + batch_size]
        batch_embeddings = model.encode(batch_texts, batch_size=batch_size, show_progress_bar=False, convert_to_tensor=True)
        embeddings.append(batch_embeddings)
    return torch.cat(embeddings, dim=0)


import torch
import numpy as np
import pandas as pd
import gc
from tqdm import tqdm
from sentence_transformers import util

def bootstrap_scores_with_adjustment(df, model, evidence_words, intuition_words, text_col, id_col, batch, iterations=1000):
    print("Embedding Dictionaries")

    # Compute embeddings for all evidence and intuition words
    evidence_embeddings = get_embeddings(evidence_words, model)
    intuition_embeddings = get_embeddings(intuition_words, model)

    print("Embedding Corpus")
    transcript_embeddings = get_embeddings_batched(df[text_col].tolist(), model, batch_size=batch)

    print("Move to GPU")
    # Ensure embeddings are on the same device as the model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    evidence_embeddings = evidence_embeddings.to(device)
    intuition_embeddings = intuition_embeddings.to(device)
    transcript_embeddings = transcript_embeddings.to(device)

    print("Start Bootstrapping")
    num_transcripts = len(transcript_embeddings)

    # Pre-allocate NumPy arrays for statistics
    evidence_all = np.zeros((num_transcripts, iterations), dtype=np.float32)
    intuition_all = np.zeros((num_transcripts, iterations), dtype=np.float32)

    with torch.no_grad():  # Disable gradient computation
        for iter_idx in tqdm(range(iterations), desc="Bootstrapping iterations"):
            # Randomly sample words from dictionaries
            sample_size_evidence = np.random.randint(30, len(evidence_embeddings))
            sample_size_intuition = np.random.randint(30, len(intuition_embeddings))

            evidence_sample_emb = evidence_embeddings[np.random.choice(len(evidence_embeddings), sample_size_evidence, replace=False)].mean(dim=0)
            intuition_sample_emb = intuition_embeddings[np.random.choice(len(intuition_embeddings), sample_size_intuition, replace=False)].mean(dim=0)

            # Compute cosine similarity for all transcripts at once
            evidence_scores = util.cos_sim(transcript_embeddings, evidence_sample_emb.unsqueeze(0)).squeeze(1)
            intuition_scores = util.cos_sim(transcript_embeddings, intuition_sample_emb.unsqueeze(0)).squeeze(1)

            # Store scores in pre-allocated NumPy arrays
            evidence_all[:, iter_idx] = evidence_scores.cpu().numpy()
            intuition_all[:, iter_idx] = intuition_scores.cpu().numpy()

    print("Computing Summary Statistics")

    # Compute summary statistics in a vectorized way
    evidence_means = np.mean(evidence_all, axis=1)
    evidence_medians = np.median(evidence_all, axis=1)
    evidence_ci_low = np.percentile(evidence_all, 2.5, axis=1)
    evidence_ci_high = np.percentile(evidence_all, 97.5, axis=1)

    intuition_means = np.mean(intuition_all, axis=1)
    intuition_medians = np.median(intuition_all, axis=1)
    intuition_ci_low = np.percentile(intuition_all, 2.5, axis=1)
    intuition_ci_high = np.percentile(intuition_all, 97.5, axis=1)

    # Create the final DataFrame
    results_main = pd.DataFrame({
        id_col: df[id_col].values,  # Keep original order
        'evidence_mean': evidence_means,
        'evidence_median': evidence_medians,
        'evidence_ci_low': evidence_ci_low,
        'evidence_ci_high': evidence_ci_high,
        'intuition_mean': intuition_means,
        'intuition_median': intuition_medians,
        'intuition_ci_low': intuition_ci_low,
        'intuition_ci_high': intuition_ci_high
    })

    # Memory cleanup before returning
    print("Cleaning up memory...")
    del transcript_embeddings, evidence_embeddings, intuition_embeddings
    del evidence_all, intuition_all
    torch.cuda.empty_cache()  # Free GPU memory
    gc.collect()  # Free CPU memory

    print("Returning DataFrame")
    print(results_main.head())
    return results_main



def main(df, model, text_col, id_col, batch):
    tqdm.pandas()

    evidence_keywords = pd.read_csv("/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv")['evidence'].dropna().tolist()
    intuition_keywords = pd.read_csv("/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv")['intuition'].dropna().tolist()

    # Perform bootstrapping with adjustment
    bootstrapped_main = bootstrap_scores_with_adjustment(df, model, evidence_keywords, intuition_keywords, text_col, id_col, batch)

    # Merge results back into the main DataFrame
    df = pd.merge(df, bootstrapped_main, on=id_col)

    return df

In [3]:
df = pd.read_csv("/kaggle/input/final-data/videos_all.csv")

model = SentenceTransformer("/kaggle/input/sbert-model-yt/yt_model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

df = main(df, model, "transcript_clean", "video_id", 100)

df.to_csv("videos_all.csv", index = False)


del df, model

Embedding Dictionaries


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding Corpus


Encoding batches: 100%|██████████| 76/76 [00:27<00:00,  2.72it/s]


Move to GPU
Start Bootstrapping


Bootstrapping iterations: 100%|██████████| 1000/1000 [00:01<00:00, 964.38it/s]


Computing Summary Statistics
Cleaning up memory...
Returning DataFrame
      video_id  evidence_mean  evidence_median  evidence_ci_low  \
0  Kzhl7pNVLNU       0.356105         0.358407         0.302349   
1  TG1X7uasgLE       0.293421         0.294791         0.249466   
2  1XyTSjwRLng       0.466760         0.468798         0.421879   
3  H2T5ZDAhOsE       0.344336         0.347155         0.289474   
4  -tZ2tK4_G5M       0.450786         0.453243         0.397804   

   evidence_ci_high  intuition_mean  intuition_median  intuition_ci_low  \
0          0.398488        0.268399          0.269161          0.232777   
1          0.331608        0.358405          0.359468          0.330228   
2          0.500946        0.311595          0.312757          0.282320   
3          0.390918        0.311329          0.312879          0.277885   
4          0.492233        0.369200          0.370733          0.331229   

   intuition_ci_high  
0           0.300968  
1           0.383496  
2     

In [4]:
df = pd.read_csv("/kaggle/input/final-data/twitter.csv")

model = SentenceTransformer("/kaggle/input/sbert-twitter/twitter_model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

df = main(df, model, "text_clean", "id", 10000)

df.to_csv("twitter.csv", index = False)


del df, model

Embedding Dictionaries


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding Corpus


Encoding batches: 100%|██████████| 93/93 [00:30<00:00,  3.05it/s]


Move to GPU
Start Bootstrapping


Bootstrapping iterations: 100%|██████████| 1000/1000 [00:56<00:00, 17.59it/s]


Computing Summary Statistics
Cleaning up memory...
Returning DataFrame
                    id  evidence_mean  evidence_median  evidence_ci_low  \
0  1503680985224949766       0.257829         0.259841         0.220927   
1  1503325058781044739       0.222704         0.223768         0.181346   
2  1503324790811205635       0.231332         0.233048         0.194470   
3  1503071011461283841       0.242340         0.243803         0.205983   
4  1503030431985328137       0.197966         0.199631         0.157984   

   evidence_ci_high  intuition_mean  intuition_median  intuition_ci_low  \
0          0.288133        0.189286          0.189804          0.159377   
1          0.258611        0.208587          0.209486          0.168067   
2          0.261822        0.198515          0.199202          0.167374   
3          0.274205        0.192672          0.193657          0.157489   
4          0.229271        0.154209          0.155962          0.115626   

   intuition_ci_high  
0   

In [5]:
df = pd.read_csv("/kaggle/input/final-data/speeches.csv")

model = SentenceTransformer("/kaggle/input/sbert-model-new/model")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

df = main(df, model, "text", "id", 5000)

df.to_csv("speeches.csv", index = False)


del df, model

/tmp/ipykernel_24/4290753582.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/kaggle/input/final-data/speeches.csv")


Embedding Dictionaries


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding Corpus


Encoding batches: 100%|██████████| 17/17 [00:13<00:00,  1.24it/s]


Move to GPU
Start Bootstrapping


Bootstrapping iterations: 100%|██████████| 1000/1000 [00:04<00:00, 234.55it/s]


Computing Summary Statistics
Cleaning up memory...
Returning DataFrame
       id  evidence_mean  evidence_median  evidence_ci_low  evidence_ci_high  \
0  795232       0.269292         0.272007         0.208647          0.321053   
1  794922       0.263692         0.267270         0.207493          0.310554   
2  794974       0.257094         0.259765         0.201871          0.299925   
3  794931       0.245022         0.246849         0.196869          0.283522   
4  794925       0.205275         0.207798         0.156988          0.247108   

   intuition_mean  intuition_median  intuition_ci_low  intuition_ci_high  
0        0.415130          0.416180          0.373133           0.454116  
1        0.023100          0.023338         -0.031360           0.076506  
2       -0.013649         -0.013928         -0.056587           0.033840  
3       -0.068317         -0.067780         -0.103399          -0.034303  
4        0.133189          0.133730          0.081401           0.185377 